# Task 3 - Manual (Human) Evaluation

We open the `assignment_01.xlsx` created in Task 2 and perform manual evaluation.

## 1. Cost Calculation

Nebius Token Factory pricing for `google/gemma-2-9b-it` (base flavor):
- Input tokens: **$0.03 per 1M tokens**
- Output tokens: **$0.09 per 1M tokens**

Note: input and output tokens are priced differently — output is 3x more expensive.

In [ ]:
import pandas as pd

df = pd.read_excel("assignment_01.xlsx")

# Nebius pricing (per token)
INPUT_PRICE  = 0.03 / 1_000_000   # $0.03 per 1M input tokens
OUTPUT_PRICE = 0.09 / 1_000_000   # $0.09 per 1M output tokens

df["cost_usd"] = df["input_tokens"] * INPUT_PRICE + df["output_tokens"] * OUTPUT_PRICE

print(f"Mean cost per call: ${df['cost_usd'].mean():.6f}")
print(f"Total cost (50 products): ${df['cost_usd'].sum():.6f}")
print(f"Mean input tokens: {df['input_tokens'].mean():.0f}")
print(f"Mean output tokens: {df['output_tokens'].mean():.0f}")

## 2. Rate Each Criterion

We selected 15 products (rows 0, 2, 3, 4, 7, 10, 11, 13, 15, 16, 18, 22, 27, 43, 49) and rated each criterion as good / ok / bad according to the Task 1 rubric.

For each product, we reviewed:
- **Fluency**: Is the text clear and easy to follow?
- **Grammar**: Any spelling, punctuation, or tense errors?
- **Tone**: Does it sound like a friendly, credible sales description?
- **Length**: Is the word count between 50-90?
- **Grounding**: Does every factual claim come from the input data?
- **Latency**: Is the response time ≤1500ms (good), ≤3000ms (ok), or >3000ms (bad)?
- **Cost**: Is the cost per call within the expected range for this model?

In [ ]:
# Manual evaluation scores for 15 selected products
evaluations = {
    0:  {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"good","latency_rating":"good","cost_rating":"good","final_score":"pass"},
    2:  {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    3:  {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    4:  {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    7:  {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    10: {"fluency":"good","grammar":"good","tone":"good","length":"ok",  "grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    11: {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"good","latency_rating":"good","cost_rating":"good","final_score":"pass"},
    13: {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"good","latency_rating":"good","cost_rating":"good","final_score":"pass"},
    15: {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    16: {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    18: {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"good","latency_rating":"good","cost_rating":"good","final_score":"pass"},
    22: {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    27: {"fluency":"good","grammar":"good","tone":"ok",  "length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    43: {"fluency":"good","grammar":"good","tone":"good","length":"good","grounding":"ok",  "latency_rating":"good","cost_rating":"good","final_score":"pass"},
    49: {"fluency":"good","grammar":"good","tone":"ok",  "length":"good","grounding":"good","latency_rating":"good","cost_rating":"good","final_score":"pass"},
}

# Apply to DataFrame
for col in ["fluency","grammar","tone","length","grounding","latency_rating","cost_rating","final_score"]:
    df[col] = df[col].astype(object)

for idx, ratings in evaluations.items():
    for col, val in ratings.items():
        df.at[idx, col] = val

df.to_excel("assignment_01.xlsx", index=False)
print("Saved evaluations to assignment_01.xlsx")

## 3. Final Score

Applied the Task 1 pass/fail rules:
- **Auto-fail** if Grounding, Length, Fluency, or Grammar = bad
- **Pass** if G ≥ 4, O ≤ 2, B = 0

Result: **15/15 pass** — no descriptions triggered automatic failure or fell below the cumulative bar.

## 4. Baseline Analysis

### Score Distribution (15 evaluated products)

| Criterion | Good | Ok | Bad |
|-----------|------|----|-----|
| Fluency | 15 | 0 | 0 |
| Grammar | 15 | 0 | 0 |
| Tone | 13 | 2 | 0 |
| Length | 14 | 1 | 0 |
| Grounding | 5 | 10 | 0 |
| Latency | 15 | 0 | 0 |
| Cost | 15 | 0 | 0 |

### Best Performing Criteria

**Fluency, Grammar, Latency, and Cost** all scored 15/15 good. The model produces well-written, grammatically correct text quickly and cheaply.

### Worst Performing Criterion

**Grounding** is by far the weakest — only 5/15 good, with 10/15 ok. The model consistently adds plausible-but-not-given details:

1. **Expanding abbreviations** — "7-in-1 multicooker" gets expanded into listing all 7 specific functions
2. **Adding brand-specific knowledge** — "1080p camera" becomes "1080p FaceTime camera"
3. **Inferring material properties** — "nickel coating" becomes "helps dissipate heat"
4. **Inventing duration claims** — "vacuum insulated" becomes "keeps drinks hot/cold for hours"
5. **Adding use cases not in data** — "synthetic leather earcups" leads to mentions of "podcasts, audiobooks"

### Improvement Strategy

Since grounding is the clear weak point, Task 4 will focus on:
- Rewriting the system prompt with explicit anti-hallucination rules targeting these 5 patterns
- Lowering temperature to reduce creative embellishment
- Adding a second few-shot example that demonstrates restraint